In [8]:
import os, random
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from sklearn.metrics import roc_auc_score, average_precision_score


In [9]:

GRAPH_DIR = r"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph"
OUT_DIR = os.path.join(GRAPH_DIR, "pipeline_outputs")
os.makedirs(OUT_DIR, exist_ok=True)


DEVICE = torch.device("cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

EPOCHS = 6
BATCH_SIZE = 2048
NEG_RATIO = 1
MC_RUNS = 30
TOP_K = 50
PATH_REG_WEIGHT = 0.5

In [3]:

def safe_read_lines(path):
    if not os.path.exists(path):
        return []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return [ln.rstrip("\n") for ln in f]

def read_csv_triples(path):
    if not os.path.exists(path):
        return []
    df = pd.read_csv(path, header=None, dtype=str)
    if df.shape[1] < 3:
        return []
    return [(r[0].strip(), r[1].strip(), r[2].strip()) for r in df.values if len(r)>=3]


In [4]:



edge_index = torch.load(os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/edge_index.pt")).long()
edge_type = torch.load(os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/edge_type.pt")).long() if os.path.exists(os.path.join(GRAPH_DIR,"edge_type.pt")) else None
num_nodes = int(edge_index.max().item()) + 1

entities_lines = safe_read_lines(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/entities.txt"))
relation_lines = safe_read_lines(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/relations.txt"))

train_triples = read_csv_triples(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/train_inductive.csv"))
val_triples   = read_csv_triples(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/val_inductive.csv"))
test_triples  = read_csv_triples(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/test_inductive.csv"))



In [5]:

# Collect all unique entities from all triples
all_entities = set()
for triple_list in [train_triples, val_triples, test_triples]:
    for h, r, t in triple_list:
        all_entities.update([h, t])

all_entities = list(all_entities)
ent2idx = {e: i for i, e in enumerate(all_entities)}
idx2ent = {i: e for e, i in ent2idx.items()}

# Assign types based on ID prefixes
ent2type = {}
for e in all_entities:
    if e.startswith("Compound::"):
        ent2type[e] = "compound"
    elif e.startswith("Gene::"):
        ent2type[e] = "gene"
    elif e.startswith("Disease::"):
        ent2type[e] = "disease"
    else:
        ent2type[e] = "other"

num_nodes = len(all_entities)  # number of embeddings


In [6]:


def build_cd_pairs(triples):
    """Return list of (compound_idx, disease_idx, label)"""
    pairs = []
    for h, r, t in triples:
        h_type = ent2type.get(h, "")
        t_type = ent2type.get(t, "")
        if ("compound" in h_type and "disease" in t_type) or "treat" in r.lower():
            pairs.append((ent2idx[h], ent2idx[t], 1))  # positive sample
    return pairs

train_pairs = build_cd_pairs(train_triples)
val_pairs = build_cd_pairs(val_triples)
test_pairs = build_cd_pairs(test_triples)

print(f"Train pairs: {len(train_pairs)}")
print(f"Val pairs: {len(val_pairs)}")
print(f"Test pairs: {len(test_pairs)}")


Train pairs: 61954
Val pairs: 6842
Test pairs: 15099


In [7]:
comp_gene_pairs = set()
gene_disease_pairs = set()
for h, r, t in train_triples:
    h_type = ent2type.get(h, "other")
    t_type = ent2type.get(t, "other")
    if h_type == "compound" and t_type == "gene":
        comp_gene_pairs.add((ent2idx[h], ent2idx[t]))
    if h_type == "gene" and t_type == "disease":
        gene_disease_pairs.add((ent2idx[h], ent2idx[t]))

print(f"Compound-Gene mechanistic pairs: {len(comp_gene_pairs)}")
print(f"Gene-Disease mechanistic pairs: {len(gene_disease_pairs)}")



Compound-Gene mechanistic pairs: 133465
Gene-Disease mechanistic pairs: 74020


In [10]:
#  GCN 
class GCNModel(nn.Module):
    def __init__(self, num_nodes, hidden_dim=128, out_dim=128):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)

    def forward(self, edge_index):
        x = self.emb.weight.to(edge_index.device)
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

# GAT 
class GATModel(nn.Module):
    def __init__(self, num_nodes, hidden_dim=128, out_dim=128, heads=4):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = GATConv(hidden_dim, hidden_dim, heads=heads, concat=True)
        self.conv2 = GATConv(hidden_dim*heads, out_dim, heads=1, concat=False)

    def forward(self, edge_index):
        x = self.emb.weight.to(edge_index.device)
        x = F.elu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

# GraphSAGE 
class GraphSAGEModel(nn.Module):
    def __init__(self, num_nodes, hidden_dim=128, out_dim=128):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = SAGEConv(hidden_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)

    def forward(self, edge_index):
        x = self.emb.weight.to(edge_index.device)
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

In [11]:

#  NEGATIVE SAMPLING 
def add_negative_samples(pairs, num_nodes, neg_ratio=1):
    neg_pairs = []
    pos_set = set((c,d) for c,d,l in pairs)
    all_nodes = list(range(num_nodes))
    for c,d,_ in pairs:
        for _ in range(neg_ratio):
            d_neg = random.choice(all_nodes)
            if (c,d_neg) not in pos_set:
                neg_pairs.append((c,d_neg,0))
    return pairs + neg_pairs


In [12]:

#  HITS@K 
def hits_at_k(y_true, y_score, k=50):
    order = np.argsort(-y_score)
    y_true_sorted = np.array(y_true)[order]
    return min(1.0, y_true_sorted[:k].sum() / k)



In [13]:

# TRAIN FUNCTION 
def train_model(model, pairs, edge_index, comp_gene_pairs, gene_disease_pairs, lr=1e-3, epochs=EPOCHS, batch_size=BATCH_SIZE):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(pairs)
    
    for epoch in range(epochs):
        random.shuffle(pairs)
        total_loss = 0
        model.train()
        for i in range(0, n, batch_size):
            batch = pairs[i:i+batch_size]
            comps = [c for c,d,l in batch]
            dises = [d for c,d,l in batch]
            labels = torch.tensor([l for _,_,l in batch], dtype=torch.float, device=DEVICE)
            
            embs = model(edge_index.to(DEVICE))
            c_embs = embs[torch.tensor(comps, device=DEVICE)]
            d_embs = embs[torch.tensor(dises, device=DEVICE)]
            logits = (c_embs*d_embs).sum(dim=1)
            loss = F.binary_cross_entropy_with_logits(logits, labels)
            
            # --- mechanistic path regularization ---
            path_loss = mechanistic_path_loss(embs, comp_gene_pairs, gene_disease_pairs)
            total_batch_loss = loss + PATH_REG_WEIGHT * path_loss
            
            optimizer.zero_grad()
            total_batch_loss.backward()
            optimizer.step()
            
            total_loss += total_batch_loss.item()
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/n:.6f}")



In [14]:
#  PREDICT WITH MC DROPOUT 
@torch.no_grad()
def predict_with_uncertainty(model, edge_index, pairs, mc_runs=MC_RUNS):
    model.eval()
    comps = [c for c,d,l in pairs]
    dises = [d for c,d,l in pairs]
    logits_list = []
    
    for _ in range(mc_runs):
        embs = model(edge_index.to(DEVICE))
        c_embs = embs[torch.tensor(comps, device=DEVICE)]
        d_embs = embs[torch.tensor(dises, device=DEVICE)]
        logits = (c_embs*d_embs).sum(dim=1)
        logits_list.append(torch.sigmoid(logits).cpu().numpy())
        
    all_logits = np.stack(logits_list)
    mean_preds = all_logits.mean(axis=0)
    std_preds = all_logits.std(axis=0)
    return mean_preds, std_preds


In [15]:

#  EVALUATE MODEL
def evaluate_model(model, edge_index, pairs):
    mean_preds, std_preds = predict_with_uncertainty(model, edge_index, pairs)
    y_true = [l for _,_,l in pairs]
    metrics = {
        "ROC-AUC": roc_auc_score(y_true, mean_preds) if len(set(y_true))>1 else float('nan'),
        "AP": average_precision_score(y_true, mean_preds),
        "Hits@50": hits_at_k(y_true, mean_preds, k=TOP_K)
    }
    return metrics, mean_preds, std_preds


In [16]:

#SAVE RANKED CSV
def save_ranked_csv(pairs, mean_preds, std_preds, outpath):
    df = pd.DataFrame({
        "compound": [c for c,d,l in pairs],
        "disease": [d for c,d,l in pairs],
        "score": mean_preds,
        "uncertainty": std_preds
    })
    df.sort_values("score", ascending=False, inplace=True)
    df.to_csv(outpath, index=False)
    print(f"Saved ranked CSV: {outpath}")


In [17]:

#  TRAIN MULTIPLE MODELS
def train_multiple_models(models_dict, train_pairs, val_pairs, test_pairs, edge_index, comp_gene_pairs, gene_disease_pairs, epochs=EPOCHS):
    results = {}
    for name, model in models_dict.items():
        print(f"\n=== Training {name} ===")
        train_model(model, train_pairs, edge_index, comp_gene_pairs, gene_disease_pairs, epochs=epochs)
        
        print(f"--- Evaluating {name} on Validation ---")
        val_metrics, val_mean, val_std = evaluate_model(model, edge_index, val_pairs)
        save_ranked_csv(val_pairs, val_mean, val_std, f"{OUT_DIR}/{name}_ranked_val.csv")
        
        print(f"--- Evaluating {name} on Test ---")
        test_metrics, test_mean, test_std = evaluate_model(model, edge_index, test_pairs)
        save_ranked_csv(test_pairs, test_mean, test_std, f"{OUT_DIR}/{name}_ranked_test.csv")
        
        results[name] = {"val": val_metrics, "test": test_metrics}
        print(f"Validation metrics: {val_metrics}")
        print(f"Test metrics: {test_metrics}")
    return results


In [ ]:

if __name__=="__main__":
    # Apply negative sampling
    train_pairs = add_negative_samples(train_pairs, num_nodes, NEG_RATIO)
    val_pairs = add_negative_samples(val_pairs, num_nodes, NEG_RATIO)
    test_pairs = add_negative_samples(test_pairs, num_nodes, NEG_RATIO)

    print(f"Train pairs: {len(train_pairs)}")
    print(f"Val pairs: {len(val_pairs)}")
    print(f"Test pairs: {len(test_pairs)}")

    # Define models
    models = {
    "GCN": GCNModel(num_nodes, hidden_dim=128, out_dim=128),
    "GAT": GATModel(num_nodes, hidden_dim=128, out_dim=128, heads=4), 
    "GraphSAGE": GraphSAGEModel(num_nodes, hidden_dim=128, out_dim=128),  
}

    results = train_multiple_models(models, train_pairs, val_pairs, test_pairs,
                                    edge_index, comp_gene_pairs, gene_disease_pairs,
                                    epochs=EPOCHS)


Train pairs: 1965503
Val pairs: 218705
Test pairs: 481903

=== Training GCN2 ===
